In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import os

In [ ]:
file_path: str = "./test_data/1.wav"
sample_rate: int = 16000

def get_band_energy(y, sr, low_freq=300, high_freq=1000):
    """Computes the energy in a specific frequency band."""
    fft_result = np.fft.fft(y)
    freqs = np.fft.fftfreq(len(y), 1/sr)
    band_energy = np.sum(np.abs(fft_result[(freqs >= low_freq) & (freqs <= high_freq)]))
    return band_energy

def show_waveform(
    file_path: str, 
    sample_rate: int, 
    segment_duration: float = 0.5,
    energy_threshold_factor: float = 0.8,
    amplitude_threshold: float = 0.02,
    ):

    # Load the audio file
    y, sr = librosa.load(file_path, sr=sample_rate)
    segment_samples = int(sr * segment_duration)
    detected_segments = []

    # Set the initial energy threshold
    initial_energy = get_band_energy(y[:segment_samples], sr)
    energy_threshold = initial_energy * energy_threshold_factor

    # Identify speech segments
    for i in range(0, len(y), segment_samples):
        y_segment = y[i:i + segment_samples]
        if len(y_segment) < segment_samples:
            break
        energy = get_band_energy(y_segment, sr)
        amplitude = np.max(np.abs(y_segment))
        if energy > energy_threshold and amplitude > amplitude_threshold:
            detected_segments.append((i / sr, (i + segment_samples) / sr, "speaker"))

    print("Detected Segments:", detected_segments)  
    
    return y, sr, detected_segments

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualize_segment(y, sr, detected_segments, time_unit: str = "minutes"):
    """
    Visualizes the waveform with detected speech segments, handling multiple segment types.
    
    Args:
        y (numpy.ndarray): Audio waveform data.
        sr (int): Sample rate of the audio.
        detected_segments (list of tuples): List of detected segments [(start, end, label), ...].
        time_unit (str, optional): Time scale ("minutes" or "seconds"). Default is "minutes".
    """
    # Visualization setup
    plt.figure(figsize=(12, 4))

    # Convert time axis based on the selected unit
    if time_unit == "minutes":
        time_axis = np.linspace(0, len(y) / sr / 60, len(y))  
        unit_label = "Time (minutes)"
        detected_segments = [(start / 60, end / 60, label) for start, end, label in detected_segments]
    else:
        time_axis = np.linspace(0, len(y) / sr, len(y))  
        unit_label = "Time (seconds)"

    plt.plot(time_axis, y, alpha=0.7, label="Waveform")

    # Define colors for different segment types
    colors = {
        "speaker": "gray",
        "tank0": "red",
        "tank1": "blue",
        "tank2": "green",
        "tank3": "purple"
    }

    # Extract unique labels for legend
    unique_labels = set(label for _, _, label in detected_segments)

    # Highlight detected speech segments with different colors
    for start, end, label in detected_segments:
        color = colors.get(label, "gray")  # Default to gray if label is not in dictionary
        plt.axvspan(start, end, color=color, alpha=0.3)

    # Create a legend with different segment labels
    legend_patches = [mpatches.Patch(color=colors[label], label=label) for label in unique_labels]
    plt.legend(handles=legend_patches)

    # Configure the graph
    plt.xlabel(unit_label)  
    plt.ylabel("Amplitude")
    plt.title("Waveform with Detected Speech Segments")
    plt.grid()
    plt.show()


y, sr, seg = show_waveform(file_path, sample_rate)
visualize_segment(y, sr, seg)

In [ ]:
def save_as_npy(y, sr, seg, npy_path):
    data = {
        'audio': y,
        'sr': sr,
        'segments': seg
    }
    print(data)
    np.save(npy_path, data, allow_pickle=True)
    print(f'Saved NPY: {npy_path}')


save_as_npy(y, sr, seg, './1.npy')

In [ ]:
data = np.load('./1.npy', allow_pickle=True).item()

print(data.keys())
print(data['segments'])
visualize_segment(data['audio'], data['sr'], data['segments'])

### **Slice audio and segments**

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt


def slice_audio(y, sr, slice_duration=5):
    """
    Slices audio into fixed-duration segments.
    
    Args:
        y (numpy.ndarray): Audio waveform data.
        sr (int): Sample rate of the audio.
        slice_duration (int): Duration of each slice in seconds (default: 5).
        
    Returns:
        list: List of sliced audio segments.
    """
    slice_samples = int(sr * slice_duration)
    slices = [y[i:i + slice_samples] for i in range(0, len(y), slice_samples)]
    return slices

def slice_segments(segments, slice_duration=5):
    """
    Adjusts detected speech segments to match sliced audio.
    
    Args:
        segments (list of tuples): List of detected speech segments in the format (start, end, label).
        slice_duration (int): Duration of each sliced segment in seconds.
        
    Returns:
        dict: Dictionary where keys are slice indices and values are lists of adjusted segments.
    """
    sliced_segments = {}
    
    for start, end, label in segments:
        slice_index = int(start // slice_duration)  # Determine which slice the segment belongs to
        relative_start = start - (slice_index * slice_duration)  # Adjust start time within the slice
        relative_end = end - (slice_index * slice_duration)  # Adjust end time within the slice
        
        if slice_index not in sliced_segments:
            sliced_segments[slice_index] = []
        
        # Ensure the segment remains within the slice boundary
        if relative_start < slice_duration:
            sliced_segments[slice_index].append((max(0, relative_start), min(slice_duration, relative_end), label))

    return sliced_segments

# --- Run the waveform analysis to detect speech segments ---
file_path: str = "/home/kar/Projects/EEND_SelfAttention/EEND_SelfAttention/data/test_data/0.wav"
sample_rate: int = 16000

y, sr, detected_segments = show_waveform(file_path, sample_rate)

# --- Slice audio and detected segments into 5-second intervals ---
sliced_audio = slice_audio(y, sr, slice_duration=5)
sliced_segments = slice_segments(detected_segments, slice_duration=5)

# --- Output results for verification ---
print(f"Total Slices: {len(sliced_audio)}")
# for i, segs in sliced_segments.items():
#     print(f"Slice {i}: {segs}")

# --- Function to visualize a specific sliced segment ---
def visualize_slice(slice_index, y_slices, sr, seg_slices):
    """
    Visualizes a specific sliced waveform with detected speech segments.
    
    Args:
        slice_index (int): The index of the slice to visualize.
        y_slices (list of numpy.ndarray): List of sliced audio waveforms.
        sr (int): Sample rate of the audio.
        seg_slices (dict): Dictionary containing detected speech segments for each slice.
    """
    plt.figure(figsize=(12, 4))
    
    y_slice = y_slices[slice_index]  # Get the waveform of the selected slice
    time_axis = np.linspace(0, len(y_slice) / sr, len(y_slice))  
    plt.plot(time_axis, y_slice, alpha=0.7, label="Waveform")

    # Highlight detected speech segments within the slice
    if slice_index in seg_slices:
        for start, end, _ in seg_slices[slice_index]:
            plt.axvspan(start, end, color='red', alpha=0.3)

    # Configure graph labels and title
    plt.xlabel("Time (seconds)")  
    plt.ylabel("Amplitude")
    plt.title(f"Waveform - Slice {slice_index}")
    plt.legend(["Waveform", "Detected Speech"])
    plt.grid()
    plt.show()


# --- Example: Visualizing the first sliced segment ---
visualize_slice(0, sliced_audio, sr, sliced_segments)

In [ ]:
import numpy as np
import librosa
import soundfile as sf

def merge_audio_with_labeled_segments(file1, segments1, file2, segments2, output_file, sample_rate=16000, mode="overlay"):
    """
    Merges two audio files while distinguishing speech segments with different labels.
    
    Args:
        file1 (str): Path to the first audio file.
        segments1 (list of tuples): Speech segments for the first file [(start, end, label), ...].
        file2 (str): Path to the second audio file.
        segments2 (list of tuples): Speech segments for the second file [(start, end, label), ...].
        output_file (str): Path to save the merged audio.
        sample_rate (int, optional): Desired sample rate for both files (default: 16000).
        mode (str, optional): "concatenate" to merge sequentially, "overlay" to mix (default: "overlay").
        
    Returns:
        numpy.ndarray: Merged audio waveform.
        list: Merged detected speech segments with labeled speakers.
    """
    # Load both audio files with the same sample rate
    y1, sr1 = librosa.load(file1, sr=sample_rate)
    y2, sr2 = librosa.load(file2, sr=sample_rate)

    if mode == "overlay":
        # Pad the shorter audio so both are the same length
        max_len = max(len(y1), len(y2))
        y1 = np.pad(y1, (0, max_len - len(y1)))
        y2 = np.pad(y2, (0, max_len - len(y2)))

        # Mix both audio files together
        merged_audio = y1 + y2
        merged_audio = merged_audio / np.max(np.abs(merged_audio))  # Normalize

        # Merge segments, keeping the original timing but renaming labels
        merged_segments = [(start, end, "tank0") for start, end, _ in segments1] + \
                          [(start, end, "tank1") for start, end, _ in segments2]

    elif mode == "concatenate":
        # Concatenate the two audio files
        merged_audio = np.concatenate((y1, y2))

        # Compute the time shift for the second audio
        offset = len(y1) / sample_rate  # Convert samples to seconds
        
        # Adjust segments and rename labels
        adjusted_segments1 = [(start, end, "tank0") for start, end, _ in segments1]
        adjusted_segments2 = [(start + offset, end + offset, "tank1") for start, end, _ in segments2]
        
        merged_segments = adjusted_segments1 + adjusted_segments2

    else:
        raise ValueError("Invalid mode. Choose 'concatenate' or 'overlay'.")

    # Save the merged audio file
    sf.write(output_file, merged_audio, sample_rate)

    return merged_audio, merged_segments

# Example usage
file1 = "./test_data/0.wav"
file2 = "./test_data/1.wav"
output_file = "merged_audio.wav"

# Example detected speech segments
segments1 = np.load('./0.npy', allow_pickle=True).item()['segments']
segments2 = np.load('./1.npy', allow_pickle=True).item()['segments']

merged_audio, merged_segments = merge_audio_with_labeled_segments(file1, segments1, file2, segments2, output_file, mode="overlay")

# Print merged segments with labels
print("Merged Segments:", merged_segments)

visualize_segment(merged_audio, 16000, merged_segments)

### **Merge audio data**

In [ ]:
import numpy as np
import librosa
import soundfile as sf

def merge_multiple_audio(files, segment_files, output_file, sample_rate=16000, mode="overlay"):
    """
    Merges multiple audio files while distinguishing speech segments with different labels.
    
    Args:
        files (list of str): List of audio file paths.
        segment_files (list of str): List of corresponding segment `.npy` files.
        output_file (str): Path to save the merged audio.
        sample_rate (int, optional): Sample rate for all audio files (default: 16000).
        mode (str, optional): "concatenate" for sequential merging, "overlay" for mixing (default: "overlay").
        
    Returns:
        numpy.ndarray: Merged audio waveform.
        list: Merged detected speech segments with labeled speakers.
    """
    audio_data = []
    segment_data = []
    max_length = 0  # Track max length for overlay mode

    # Load all audio files and corresponding segments
    for i, (file, seg_file) in enumerate(zip(files, segment_files)):
        y, sr = librosa.load(file, sr=sample_rate)
        segments = np.load(seg_file, allow_pickle=True).item().get('segments', [])

        # Assign dynamic labels (tank0, tank1, tank2, ...)
        labeled_segments = [(start, end, f"tank{i}") for start, end, _ in segments]
        
        audio_data.append(y)
        segment_data.append(labeled_segments)

        max_length = max(max_length, len(y))  # Update max length for overlay mode

    if mode == "overlay":
        # Pad all audio to the same length
        padded_audio = [np.pad(y, (0, max_length - len(y))) for y in audio_data]
        merged_audio = sum(padded_audio)  # Overlay all audio tracks
        merged_audio = merged_audio / np.max(np.abs(merged_audio))  # Normalize to prevent clipping

        # Keep all segments unchanged since timing is the same
        merged_segments = [seg for segments in segment_data for seg in segments]

    elif mode == "concatenate":
        merged_audio = np.concatenate(audio_data)

        # Adjust segment timestamps for concatenation
        merged_segments = []
        current_offset = 0  # Track the cumulative time offset
        for i, (y, segments) in enumerate(zip(audio_data, segment_data)):
            adjusted_segments = [(start + current_offset, end + current_offset, f"tank{i}") for start, end, _ in segments]
            merged_segments.extend(adjusted_segments)
            current_offset += len(y) / sample_rate  # Convert samples to seconds

    else:
        raise ValueError("Invalid mode. Choose 'concatenate' or 'overlay'.")

    # Save the merged audio file
    sf.write(output_file, merged_audio, sample_rate)

    return merged_audio, merged_segments

# Example usage
files = ["./test_data/0.wav", "./test_data/1.wav", "./test_data/2.wav"]  # List of audio files
segment_files = ["./0.npy", "./1.npy", "./1.npy"]  # Corresponding segment files
output_file = "merged_audio.wav"

merged_audio, merged_segments = merge_multiple_audio(files, segment_files, output_file, mode="overlay")

# Print merged segments with labels
print("Merged Segments:", merged_segments)

# Visualize the merged waveform with labeled segments
visualize_segment(merged_audio, 16000, merged_segments)

### **Labeling All Audio Files in a Folder**

In [ ]:
import os
import numpy as np
import librosa

def get_band_energy(y, sr, low_freq=300, high_freq=1000):
    """
    Computes the energy in a specific frequency band.
    
    Args:
        y (numpy.ndarray): Audio waveform data.
        sr (int): Sample rate of the audio.
        low_freq (int, optional): Lower frequency bound (default: 300Hz).
        high_freq (int, optional): Upper frequency bound (default: 1000Hz).
        
    Returns:
        float: Computed band energy.
    """
    fft_result = np.fft.fft(y)
    freqs = np.fft.fftfreq(len(y), 1 / sr)
    band_energy = np.sum(np.abs(fft_result[(freqs >= low_freq) & (freqs <= high_freq)]))
    return band_energy

def detect_speech_segments(file_path, sample_rate=16000, segment_duration=0.5, energy_threshold_factor=0.8, amplitude_threshold=0.02):
    """
    Detects speech segments based on energy and amplitude thresholds.

    Args:
        file_path (str): Path to the audio file.
        sample_rate (int, optional): Sample rate for processing (default: 16000).
        segment_duration (float, optional): Duration of each segment in seconds (default: 0.5s).
        energy_threshold_factor (float, optional): Multiplier for energy threshold (default: 0.8).
        amplitude_threshold (float, optional): Minimum amplitude threshold (default: 0.02).

    Returns:
        list: Detected speech segments [(start_time, end_time, label), ...].
    """
    y, sr = librosa.load(file_path, sr=sample_rate)
    segment_samples = int(sr * segment_duration)
    detected_segments = []

    # Set the initial energy threshold
    initial_energy = get_band_energy(y[:segment_samples], sr)
    energy_threshold = initial_energy * energy_threshold_factor

    # Identify speech segments
    for i in range(0, len(y), segment_samples):
        y_segment = y[i:i + segment_samples]
        if len(y_segment) < segment_samples:
            break
        energy = get_band_energy(y_segment, sr)
        amplitude = np.max(np.abs(y_segment))
        if energy > energy_threshold and amplitude > amplitude_threshold:
            detected_segments.append((i / sr, (i + segment_samples) / sr, "speech"))

    return detected_segments

def process_audio_folder(folder_path, output_folder):
    """
    Processes all audio files in a folder, detects speech segments, and saves labeled segment data as .npy files.

    Args:
        folder_path (str): Path to the folder containing audio files.
        output_folder (str): Path to the folder where labeled .npy files will be saved.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)  # Create output folder if it doesn't exist

    audio_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    
    for idx, file_name in enumerate(audio_files):
        file_path = os.path.join(folder_path, file_name)
        label = f"tank0"  # Assign dynamic label based on order (tank0, tank1, tank2, ...)

        print(f"Processing {file_name} -> Label: {label}")

        # Detect speech segments
        segments = detect_speech_segments(file_path)

        # Save segment data to an .npy file
        npy_data = {'segments': [(start, end, label) for start, end, _ in segments]}
        np.save(os.path.join(output_folder, f"{idx}.npy"), npy_data)

    print("Processing complete. Labeled .npy files saved.")

# Example usage
folder_path = "./test_data"  # Folder containing .wav files
output_folder = "./test_data"  # Folder to save .npy files

process_audio_folder(folder_path, output_folder)

### **Generate Combinations of Merged Audio**

In [ ]:
import os
import itertools
import numpy as np
import librosa
import soundfile as sf

def merge_audio_with_segments(files, segments_list, output_file, sample_rate=16000, mode="overlay"):
    """
    Merges multiple audio files while maintaining labeled speech segments.

    Args:
        files (list of str): List of audio file paths.
        segments_list (list of list): List of segment lists corresponding to each file.
        output_file (str): Path to save the merged audio.
        sample_rate (int, optional): Sample rate for all audio files (default: 16000).
        mode (str, optional): "concatenate" for sequential merging, "overlay" for mixing (default: "overlay").

    Returns:
        numpy.ndarray: Merged audio waveform.
        list: Merged detected speech segments with labeled speakers.
    """
    audio_data = []
    segment_data = []
    max_length = 0  # Track max length for overlay mode

    # Load all audio files and corresponding segments
    for i, (file, segments) in enumerate(zip(files, segments_list)):
        y, sr = librosa.load(file, sr=sample_rate)

        # Assign labels dynamically (tank0, tank1, etc.)
        labeled_segments = [(start, end, f"tank{i}") for start, end, _ in segments]

        audio_data.append(y)
        segment_data.append(labeled_segments)

        max_length = max(max_length, len(y))  # Update max length for overlay mode

    if mode == "overlay":
        # Pad all audio to the same length
        padded_audio = [np.pad(y, (0, max_length - len(y))) for y in audio_data]
        merged_audio = sum(padded_audio)  # Overlay all audio tracks
        merged_audio = merged_audio / np.max(np.abs(merged_audio))  # Normalize

        # Keep original segment timings
        merged_segments = [seg for segments in segment_data for seg in segments]

    elif mode == "concatenate":
        merged_audio = np.concatenate(audio_data)

        # Adjust segment timestamps
        merged_segments = []
        current_offset = 0
        for i, (y, segments) in enumerate(zip(audio_data, segment_data)):
            adjusted_segments = [(start + current_offset, end + current_offset, f"tank{i}") for start, end, _ in segments]
            merged_segments.extend(adjusted_segments)
            current_offset += len(y) / sample_rate  # Convert samples to seconds

    else:
        raise ValueError("Invalid mode. Choose 'concatenate' or 'overlay'.")

    # Save the merged audio file
    sf.write(output_file, merged_audio, sample_rate)

    return merged_audio, merged_segments

def generate_combinations(folder_path, output_folder, max_merge=3, mode="overlay"):
    """
    Generates merged audio files from combinations of original audio files.

    Args:
        folder_path (str): Path to the folder containing audio files.
        output_folder (str): Path to save merged audio and labels.
        max_merge (int, optional): Maximum number of audio files to merge.
        mode (str, optional): "concatenate" or "overlay".
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)  # Create output folder if it doesn't exist

    # Get sorted list of audio files
    audio_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])
    segment_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npy')])

    # Load segment data from .npy files
    segments_dict = {file.split('.')[0]: np.load(os.path.join(folder_path, file), allow_pickle=True).item().get('segments', [])
                     for file in segment_files}

    # Process all combinations of files up to max_merge
    for num_files in range(1, max_merge + 1):
        for combo in itertools.combinations(enumerate(audio_files), num_files):
            indices, selected_files = zip(*combo)
            selected_labels = [f"tank{i}" for i in indices]

            # Load corresponding segments
            selected_segments = [segments_dict[str(i)] for i in indices]

            # Generate output filename
            merged_filename = f"merged_{'_'.join(map(str, indices))}.wav"
            output_path = os.path.join(output_folder, merged_filename)

            print(f"Merging {selected_files} -> {merged_filename}")

            # Merge audio files
            merged_audio, merged_segments = merge_audio_with_segments(
                [os.path.join(folder_path, f) for f in selected_files], 
                selected_segments, 
                output_path, 
                mode=mode
            )

            # Save segment data
            segment_output_path = os.path.join(output_folder, f"merged_{'_'.join(map(str, indices))}.npy")
            np.save(segment_output_path, {'segments': merged_segments})

    print("All combinations processed.")

# Example usage
folder_path = "./test_data"  # Folder containing original audio & .npy segment files
output_folder = "./merged_data"  # Folder to save merged audio and labels

generate_combinations(folder_path, output_folder, max_merge=3, mode="overlay")

### **Slice Audio Before Merging**

In [ ]:
import os
import itertools
import numpy as np
import librosa
import soundfile as sf

def slice_audio(y, sr, slice_duration=5):
    """
    Slices audio into fixed-duration segments.

    Args:
        y (numpy.ndarray): Audio waveform data.
        sr (int): Sample rate of the audio.
        slice_duration (int): Duration of each slice in seconds (default: 5).

    Returns:
        list: List of sliced audio segments.
    """
    slice_samples = int(sr * slice_duration)
    slices = [y[i:i + slice_samples] for i in range(0, len(y), slice_samples)]
    return slices

def slice_segments(segments, slice_duration=5):
    """
    Adjusts detected speech segments to match sliced audio.

    Args:
        segments (list of tuples): List of detected speech segments in the format (start, end, label).
        slice_duration (int): Duration of each sliced segment in seconds.

    Returns:
        dict: Dictionary where keys are slice indices and values are lists of adjusted segments.
    """
    sliced_segments = {}

    for start, end, label in segments:
        slice_index = int(start // slice_duration)  # Determine which slice the segment belongs to
        relative_start = start - (slice_index * slice_duration)  # Adjust start time within the slice
        relative_end = end - (slice_index * slice_duration)  # Adjust end time within the slice

        if slice_index not in sliced_segments:
            sliced_segments[slice_index] = []

        # Ensure the segment remains within the slice boundary
        if relative_start < slice_duration:
            sliced_segments[slice_index].append((max(0, relative_start), min(slice_duration, relative_end), label))

    return sliced_segments

def process_and_save_sliced_audio(folder_path, output_folder, slice_duration=5, sample_rate=16000):
    """
    Processes all audio files in a folder, slices them into fixed-length segments, and saves them.

    Args:
        folder_path (str): Path to the folder containing original audio files.
        output_folder (str): Path to save the sliced audio and segments.
        slice_duration (int, optional): Duration of each slice in seconds (default: 5).
        sample_rate (int, optional): Sample rate for processing (default: 16000).
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    audio_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.wav')])

    for file_name in audio_files:
        file_path = os.path.join(folder_path, file_name)
        npy_path = file_path.replace(".wav", ".npy")

        print(f"Processing {file_name}")

        # Load the audio file
        y, sr = librosa.load(file_path, sr=sample_rate)

        # Load segment data
        if os.path.exists(npy_path):
            segment_data = np.load(npy_path, allow_pickle=True).item().get("segments", [])
        else:
            segment_data = []

        # Slice the audio and corresponding segments
        sliced_audio = slice_audio(y, sr, slice_duration)
        sliced_segments = slice_segments(segment_data, slice_duration)

        # Save the sliced audio and segments
        for i, (audio_slice, segments) in enumerate(zip(sliced_audio, sliced_segments.values())):
            slice_filename = f"{file_name.replace('.wav', '')}_slice{i}.wav"
            slice_npy_filename = f"{file_name.replace('.wav', '')}_slice{i}.npy"

            sf.write(os.path.join(output_folder, slice_filename), audio_slice, sample_rate)
            np.save(os.path.join(output_folder, slice_npy_filename), {"segments": segments})

    print("Slicing completed.")

def merge_audio_with_segments(files, segments_list, output_file, sample_rate=16000, mode="overlay"):
    """
    Merges multiple sliced audio files while maintaining labeled speech segments.

    Args:
        files (list of str): List of sliced audio file paths.
        segments_list (list of list): List of segment lists corresponding to each file.
        output_file (str): Path to save the merged audio.
        sample_rate (int, optional): Sample rate for all audio files (default: 16000).
        mode (str, optional): "concatenate" for sequential merging, "overlay" for mixing (default: "overlay").

    Returns:
        numpy.ndarray: Merged audio waveform.
        list: Merged detected speech segments with labeled speakers.
    """
    audio_data = []
    segment_data = []
    max_length = 0  # Track max length for overlay mode

    for i, (file, segments) in enumerate(zip(files, segments_list)):
        y, sr = librosa.load(file, sr=sample_rate)

        # Assign labels dynamically (tank0, tank1, etc.)
        labeled_segments = [(start, end, f"tank{i}") for start, end, _ in segments]

        audio_data.append(y)
        segment_data.append(labeled_segments)

        max_length = max(max_length, len(y))  # Update max length for overlay mode

    if mode == "overlay":
        # Pad all audio to the same length
        padded_audio = [np.pad(y, (0, max_length - len(y))) for y in audio_data]
        merged_audio = sum(padded_audio)  # Overlay all audio tracks
        merged_audio = merged_audio / np.max(np.abs(merged_audio))  # Normalize

        # Keep original segment timings
        merged_segments = [seg for segments in segment_data for seg in segments]

    elif mode == "concatenate":
        merged_audio = np.concatenate(audio_data)

        # Adjust segment timestamps
        merged_segments = []
        current_offset = 0
        for i, (y, segments) in enumerate(zip(audio_data, segment_data)):
            adjusted_segments = [(start + current_offset, end + current_offset, f"tank{i}") for start, end, _ in segments]
            merged_segments.extend(adjusted_segments)
            current_offset += len(y) / sample_rate  # Convert samples to seconds

    else:
        raise ValueError("Invalid mode. Choose 'concatenate' or 'overlay'.")

    # Save the merged audio file
    sf.write(output_file, merged_audio, sample_rate)

    return merged_audio, merged_segments

def generate_combinations(folder_path, output_folder, max_merge=3, mode="overlay"):
    """
    Generates merged audio files from combinations of sliced audio files.

    Args:
        folder_path (str): Path to the folder containing sliced audio files.
        output_folder (str): Path to save merged audio and labels.
        max_merge (int, optional): Maximum number of audio files to merge.
        mode (str, optional): "concatenate" or "overlay".
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    audio_files = sorted([f for f in os.listdir(folder_path) if "_slice" in f and f.endswith('.wav')])
    segment_files = sorted([f for f in os.listdir(folder_path) if "_slice" in f and f.endswith('.npy')])

    segments_dict = {file.replace(".npy", ""): np.load(os.path.join(folder_path, file), allow_pickle=True).item().get("segments", [])
                     for file in segment_files}

    for num_files in range(1, max_merge + 1):
        for combo in itertools.combinations(enumerate(audio_files), num_files):
            indices, selected_files = zip(*combo)

            selected_segments = [segments_dict[file.replace(".wav", "")] for file in selected_files]

            merged_filename = f"merged_{'_'.join(map(str, indices))}.wav"
            output_path = os.path.join(output_folder, merged_filename)

            print(f"Merging {selected_files} -> {merged_filename}")

            merged_audio, merged_segments = merge_audio_with_segments(
                [os.path.join(folder_path, f) for f in selected_files],
                selected_segments,
                output_path,
                mode=mode
            )

            segment_output_path = os.path.join(output_folder, f"merged_{'_'.join(map(str, indices))}.npy")
            np.save(segment_output_path, {'segments': merged_segments})

    print("All combinations processed.")

# --- First, slice the original audio files ---
input_folder = "./test_data"
sliced_folder = "./sliced_data"
merged_folder = "./merged_data"

process_and_save_sliced_audio(input_folder, sliced_folder, slice_duration=5)

# --- Then, generate merged combinations using sliced audio ---
generate_combinations(sliced_folder, merged_folder, max_merge=3, mode="overlay")
